# PySpark — DataFrame Transformations

Transformations create a **new DataFrame** — they are **lazy** (nothing runs until an action like `.show()` or `.count()` is called).

| Operation | Method | Purpose |
|-----------|--------|---------|
| Choose columns | `select()` | Project specific columns |
| Filter rows | `filter()` / `where()` | Keep rows matching a condition |
| Add / modify column | `withColumn()` | Create or overwrite a column |
| Remove column | `drop()` | Delete one or more columns |
| Rename column | `withColumnRenamed()` | Rename without rewriting all columns |
| Sort rows | `orderBy()` / `sort()` | Ascending or descending |
| Remove duplicates | `distinct()` | Unique rows |
| Limit rows | `limit(n)` | First N rows |

## Setup — Create Sample DataFrame

In [4]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, lower, lit, round as spark_round

# Stop any stale session so PYSPARK_PYTHON takes effect
active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("Transformations") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    ("Alice",   "Engineering", 95000, 5),
    ("Bob",     "Marketing",   72000, 3),
    ("Charlie", "Engineering", 110000, 8),
    ("Diana",   "HR",          65000, 2),
    ("Eve",     "Marketing",   85000, 6),
    ("Frank",   "Engineering", 98000, 7),
]
columns = ["name", "department", "salary", "years"]
df = spark.createDataFrame(data, columns)
df.show()
print("Schema:", df.dtypes)

+-------+-----------+------+-----+
|   name| department|salary|years|
+-------+-----------+------+-----+
|  Alice|Engineering| 95000|    5|
|    Bob|  Marketing| 72000|    3|
|Charlie|Engineering|110000|    8|
|  Diana|         HR| 65000|    2|
|    Eve|  Marketing| 85000|    6|
|  Frank|Engineering| 98000|    7|
+-------+-----------+------+-----+

Schema: [('name', 'string'), ('department', 'string'), ('salary', 'bigint'), ('years', 'bigint')]


## 1. select() — Choose Columns

Like SQL `SELECT`. You can select by name, by `col()`, or with expressions.

In [2]:
# Select by column name (string)
df.select("name", "salary").show()

# Select using col() — allows expressions
df.select(col("name"), col("salary"), col("salary") * 1.1).show()

# Rename inline with alias()
df.select(
    col("name").alias("employee"),
    col("salary").alias("annual_salary"),
    (col("salary") * 0.10).alias("bonus")
).show()

+-------+------+
|   name|salary|
+-------+------+
|  Alice| 95000|
|    Bob| 72000|
|Charlie|110000|
|  Diana| 65000|
|    Eve| 85000|
|  Frank| 98000|
+-------+------+

+-------+------+------------------+
|   name|salary|    (salary * 1.1)|
+-------+------+------------------+
|  Alice| 95000|104500.00000000001|
|    Bob| 72000|           79200.0|
|Charlie|110000|121000.00000000001|
|  Diana| 65000|           71500.0|
|    Eve| 85000| 93500.00000000001|
|  Frank| 98000|107800.00000000001|
+-------+------+------------------+

+--------+-------------+-------+
|employee|annual_salary|  bonus|
+--------+-------------+-------+
|   Alice|        95000| 9500.0|
|     Bob|        72000| 7200.0|
| Charlie|       110000|11000.0|
|   Diana|        65000| 6500.0|
|     Eve|        85000| 8500.0|
|   Frank|        98000| 9800.0|
+--------+-------------+-------+



## 2. filter() and where() — Filter Rows

`filter()` and `where()` are **identical** — use whichever reads more naturally.  
Combine conditions with `&` (AND), `|` (OR), `~` (NOT).

In [3]:
# Single condition
df.filter(col("salary") > 80000).show()

# where() — identical to filter()
df.where(col("department") == "Engineering").show()

# AND condition — must use & with parentheses
df.filter((col("salary") > 70000) & (col("years") >= 5)).show()

# OR condition
df.filter((col("department") == "Engineering") | (col("department") == "Marketing")).show()

# String filter shorthand
df.filter("salary > 90000").show()   # SQL string syntax also works

+-------+-----------+------+-----+
|   name| department|salary|years|
+-------+-----------+------+-----+
|  Alice|Engineering| 95000|    5|
|Charlie|Engineering|110000|    8|
|    Eve|  Marketing| 85000|    6|
|  Frank|Engineering| 98000|    7|
+-------+-----------+------+-----+

+-------+-----------+------+-----+
|   name| department|salary|years|
+-------+-----------+------+-----+
|  Alice|Engineering| 95000|    5|
|Charlie|Engineering|110000|    8|
|  Frank|Engineering| 98000|    7|
+-------+-----------+------+-----+

+-------+-----------+------+-----+
|   name| department|salary|years|
+-------+-----------+------+-----+
|  Alice|Engineering| 95000|    5|
|Charlie|Engineering|110000|    8|
|    Eve|  Marketing| 85000|    6|
|  Frank|Engineering| 98000|    7|
+-------+-----------+------+-----+

+-------+-----------+------+-----+
|   name| department|salary|years|
+-------+-----------+------+-----+
|  Alice|Engineering| 95000|    5|
|    Bob|  Marketing| 72000|    3|
|Charlie|Engineer

## 3. withColumn() — Add or Modify a Column

`withColumn(name, expression)` — if `name` already exists, it **overwrites** it; otherwise adds a new column.

In [ ]:
# Add new columns
df2 = df.withColumn("bonus",      col("salary") * 0.10)
df2 = df2.withColumn("dept_upper", upper(col("department")))
df2 = df2.withColumn("salary_k",   spark_round(col("salary") / 1000, 1))
df2 = df2.withColumn("is_senior",  col("years") >= 5)   # boolean column
df2.show()

# Overwrite an existing column (apply a 10% raise)
df3 = df.withColumn("salary", col("salary") * 1.10)
df3.show()

## 4. drop() — Remove Columns

In [ ]:
# Drop a single column
df.drop("years").show()

# Drop multiple columns
df.drop("years", "department").show()

# withColumnRenamed — rename without rewriting everything
df.withColumnRenamed("name", "employee_name") \
  .withColumnRenamed("salary", "annual_salary").show()

## 5. orderBy() / sort() — Sort Rows

Both are identical. Default is ascending. Use `.desc()` for descending.

In [ ]:
# Ascending (default)
df.orderBy("salary").show()

# Descending
df.orderBy(col("salary").desc()).show()

# Sort by multiple columns
df.sort("department", col("salary").desc()).show()

## 6. distinct() and limit()

In [ ]:
# Unique values in a column
df.select("department").distinct().show()

# Unique rows across all columns
df.distinct().count()

# limit() — first N rows (useful for sampling large datasets)
df.limit(3).show()

## 7. Method Chaining — Real-World ETL Pipeline

Transformations can be chained — each returns a new DataFrame.

In [ ]:
# Build an Engineering salary report — chain everything
report = (
    df
    .filter(col("department") == "Engineering")
    .withColumn("bonus",       col("salary") * 0.15)
    .withColumn("total_comp",  col("salary") + col("bonus"))
    .select(
        col("name").alias("employee"),
        col("salary"),
        col("bonus"),
        col("total_comp")
    )
    .orderBy(col("total_comp").desc())
)
report.show()
print("Total Engineering payroll: $", report.agg({"total_comp": "sum"}).collect()[0][0])

## Quick Reference

```python
from pyspark.sql.functions import col, upper, lower, lit, round

df.select("col1", "col2")                        # choose columns
df.select(col("salary") * 1.1)                   # with expression
df.filter(col("salary") > 50000)                 # filter rows
df.where(col("dept") == "Engineering")           # identical to filter
df.filter((col("a") > 1) & (col("b") < 10))     # AND
df.filter((col("a") > 1) | (col("b") < 10))     # OR
df.withColumn("bonus", col("salary") * 0.1)      # add/overwrite column
df.drop("col1", "col2")                          # remove columns
df.withColumnRenamed("old", "new")               # rename
df.orderBy(col("salary").desc())                 # sort descending
df.distinct()                                    # remove duplicate rows
df.limit(10)                                     # first 10 rows
```

## Interview Questions

1. What is the difference between a transformation and an action in PySpark?
2. Are `filter()` and `where()` different? When would you use each?
3. What does `withColumn()` do if the column name already exists?
4. How do you apply AND / OR conditions in `filter()`? Why must conditions be in parentheses?
5. What is lazy evaluation and why does PySpark use it?
6. What is the difference between `select()` and `withColumn()`?
7. How would you rename multiple columns efficiently?

## Practice Exercises

**Easy:**
1. Select only `name` and `department` columns from `df`.
2. Filter employees with more than 4 years of experience.
3. Add a column `tax` that is 30% of `salary`.

**Medium:**
4. Filter employees in Engineering OR Marketing with salary above 80,000.
5. Add a `grade` column: `"Senior"` if years >= 5, else `"Junior"` (use `when/otherwise`).
6. Drop the `years` column and rename `salary` to `annual_pay`.

**Advanced:**
7. Build a complete pipeline: filter dept = HR or Marketing, add 15% bonus, select name/salary/bonus, sort by bonus desc.
8. Find the top 3 earners across all departments.

## 8. Adaptive Query Execution (AQE)

**Adaptive Query Execution** re-optimises a running query using real runtime statistics measured
after each shuffle stage, instead of static estimates made at planning time.

| Feature | Config key | What it does |
|---|---|---|
| Master switch | `spark.sql.adaptive.enabled` | Turns AQE on/off (default **true** in Spark 3.2+) |
| Coalesce partitions | `spark.sql.adaptive.coalescePartitions.enabled` | Merges many tiny post-shuffle partitions into fewer |
| Join strategy switch | `spark.sql.autoBroadcastJoinThreshold` | Converts SortMergeJoin → BroadcastHashJoin at runtime when AQE measures one side is small enough |
| Skew-join handling | `spark.sql.adaptive.skewJoin.enabled` | Splits an oversized (hot-key) partition across multiple tasks |

Three demos below show each feature with `.explain()` plan output and partition counts as evidence.

### Demo 1 — Coalesce Shuffle Partitions

**Problem:** `spark.sql.shuffle.partitions` defaults to 200. A `groupBy` on only 50 distinct
categories still produces 200 shuffle output partitions — most of them empty. Every empty
partition maps to a wasted task.

**AQE fix:** After the shuffle stage, AQE measures actual partition sizes and coalesces
nearly-empty ones into fewer right-sized partitions.

**Evidence:**
- AQE OFF → `getNumPartitions()` = **200**
- AQE ON  → `getNumPartitions()` = **1** (all 50 category buckets fit in one coalesced partition)
- `explain()` gains an `AdaptiveSparkPlan isFinalPlan=false` wrapper when AQE is on

In [ ]:
from pyspark.sql.functions import rand, sum as spark_sum

# --- Demo 1: Coalesce Shuffle Partitions ---

sales = (
    spark.range(100_000)
    .withColumn("category", (col("id") % 50).cast("string"))
    .withColumn("amount",   (rand(seed=1) * 100).cast("double"))
)

# AQE OFF — 200 shuffle partitions even though only 50 categories exist
spark.conf.set("spark.sql.adaptive.enabled",                    "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions",                  "200")

grouped_off = sales.groupBy("category").agg(spark_sum("amount").alias("total"))
print("=== AQE OFF — no AdaptiveSparkPlan wrapper ===")
grouped_off.explain()
print("Partitions (AQE OFF):", grouped_off.rdd.getNumPartitions())   # → 200

# AQE ON — coalescePartitions merges 200 → 1
spark.conf.set("spark.sql.adaptive.enabled",                    "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

grouped_on = sales.groupBy("category").agg(spark_sum("amount").alias("total"))
print("\n=== AQE ON — AdaptiveSparkPlan isFinalPlan=false wrapper visible ===")
grouped_on.explain()
print("Partitions (AQE ON):", grouped_on.rdd.getNumPartitions())    # → 1

### Demo 2 — Runtime Join Strategy Switching (SortMergeJoin → BroadcastHashJoin)

**Problem:** The static planner chooses `SortMergeJoin` when it cannot prove at planning time
that one side fits in memory. A classic case: a `filter()` that shrinks a large table to a
handful of rows — the planner sees the pre-filter size, not the post-filter size.

**AQE fix:** With `autoBroadcastJoinThreshold` set, AQE measures the filtered side at runtime.
If it is smaller than the threshold it **promotes the join to BroadcastHashJoin on the fly**.

**Evidence:**
- AQE OFF + threshold = -1 → `explain()` shows `SortMergeJoin`
- AQE ON  + threshold = 10 MB → `explain()` shows `BroadcastHashJoin` inside `AdaptiveSparkPlan`

In [ ]:
from pyspark.sql.functions import concat

# --- Demo 2: Runtime Join Strategy Switching ---

orders = (
    spark.range(500_000)
    .withColumnRenamed("id", "order_id")
    .withColumn("status_id", (col("order_id") % 10).cast("long"))
    .withColumn("amount",    (rand(seed=2) * 500).cast("double"))
)
# Source is 10 000 rows; filter keeps only 10 — static planner sees 10 000 and picks SortMergeJoin
statuses = (
    spark.range(10_000)
    .withColumnRenamed("id", "status_id")
    .withColumn("description", concat(lit("Status_"), col("status_id")))
    .filter(col("status_id") < 10)
)

# AQE OFF + broadcast disabled → SortMergeJoin
spark.conf.set("spark.sql.adaptive.enabled",           "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

joined_off = orders.join(statuses, on="status_id", how="inner")
print("=== AQE OFF — SortMergeJoin (planner cannot see post-filter size) ===")
joined_off.explain()

# AQE ON + 10 MB threshold → runtime measurement switches to BroadcastHashJoin
spark.conf.set("spark.sql.adaptive.enabled",           "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

joined_on = orders.join(statuses, on="status_id", how="inner")
print("\n=== AQE ON — BroadcastHashJoin inside AdaptiveSparkPlan ===")
joined_on.explain()

### Demo 3 — Skew Join Awareness

**Problem:** When ~80% of rows share a single join key (`customer_id = 0`), one shuffle partition
is vastly larger than the rest. In a cluster, that one task becomes a **straggler** — the whole
stage waits for it while all other executors sit idle.

**AQE fix:** `skewJoin.enabled` detects partitions that exceed
`spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default 256 MB) **and** are more
than `skewedPartitionFactor` (5×) larger than the median partition. It splits the hot partition
and replicates the matching dimension rows to handle it in parallel.

**Local note:** The 256 MB threshold requires production-scale shuffle data. On local synthetic
data AQE is active (visible via `AdaptiveSparkPlan` and `AQEShuffleRead coalesced`) but the skew
threshold is not physically crossed, so `skew=true` and `SkewedShuffleRead` only appear on a
real cluster. The distribution table below proves the skew exists.

In [ ]:
from pyspark.sql.functions import when, desc

# --- Demo 3: Skew Join Awareness ---

spark.conf.set("spark.sql.adaptive.enabled",           "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",  "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force SortMergeJoin
spark.conf.set("spark.sql.shuffle.partitions",         "200")

# ~80% of rows land on customer_id = 0 (the hot key)
orders_skewed = (
    spark.range(200_000)
    .withColumn("customer_id",
        when(rand(seed=42) < 0.8, lit(0))
        .otherwise((rand(seed=43) * 999 + 1).cast("long")))
    .withColumn("amount", (rand(seed=44) * 100).cast("double"))
)

customers = (
    spark.range(1_000)
    .withColumnRenamed("id", "customer_id")
    .withColumn("name", concat(lit("cust_"), col("customer_id")))
)

# Prove the skew
print("=== Key distribution — hot key (id=0) holds ~80% of rows ===")
orders_skewed.groupBy("customer_id").count().orderBy(desc("count")).show(5)

# AQE plan — AdaptiveSparkPlan wrapper present; skewJoin.enabled config active
joined = orders_skewed.join(customers, on="customer_id", how="inner")
print("=== AQE ON with skewJoin.enabled — plan before execution ===")
joined.explain(mode="formatted")

print("""
[Cluster behaviour] When the hot partition exceeds 256 MB the final plan shows:
  AQEShuffleRead skewed
  SortMergeJoin (left skew=true)
confirming Spark split the skewed partition into multiple balanced tasks.
""")

In [ ]:
# --- Reset all AQE configs to Spark defaults ---
spark.conf.set("spark.sql.adaptive.enabled",                    "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",           "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",          str(10 * 1024 * 1024))
spark.conf.set("spark.sql.shuffle.partitions",                  "200")

print("AQE configs restored:")
for k in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
]:
    print(f"  {k} = {spark.conf.get(k)}")

In [ ]:
spark.stop()